# Risk visualisation for relative drought — Kazakhstan

- Adapted from the CLIMAAX [Handbook](https://handbook.climaax.eu/) and [DROUGHTS](https://github.com/CLIMAAX/DROUGHTS) GitHub repository.
- Methodology: Carrão et al. (2016) [doi:10.1016/j.gloenvcha.2016.04.012](https://doi.org/10.1016/j.gloenvcha.2016.04.012)

## Aims of the workflow

This workflow visualises and explores relative drought risk for Kazakhstan **districts**. It includes maps of relative drought risk at district level for different scenarios and WASP hazard diagnostics for a selected **focal oblast**.

:::{caution}
Districts with no ISIMIP3b grid coverage at 0.5° resolution (typically small city-districts) will appear grey on maps — this is expected behaviour. See the hazard assessment notebook for the full list.
:::

## Preliminaries

### Load libraries

In [ ]:
import os
os.environ['USE_PYGEOS'] = '0'
import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

### Define working environment and global parameters

In [ ]:
SHAPEFILE_LVL3 = Path(
    r'C:\Users\dauzo\Models\crabook-kazakhstan\crabook'
    r'\KAZ_BORDER_VERSION_1\LVL3\SHP_LVL3\KAZ_OSM_BORDER_LVL3.shp'
)  
REGION_NAME_FIELD = 'name_en'
OBLAST_NAME_FIELD = 'oblast_en'

OUTPUT_DIR = Path('./data/isimip3b/processed/outputs_hazards')
RISK_DIR   = Path('./data/isimip3b/processed/outputs_risk_districts')

data = ['historic', 'ssp126_nf', 'ssp126_ff', 'ssp370_nf', 'ssp370_ff']

print('Configuration ready.')

### Load Kazakhstan district shapefile and define regions of interest

In [ ]:
districts = gpd.read_file(SHAPEFILE_LVL3).to_crs(epsg=4326)
districts = districts.rename(columns={
    REGION_NAME_FIELD: 'district_name',
    OBLAST_NAME_FIELD: 'oblast_name',
})

districts['district_name'] = districts['district_name'].str.strip()
districts['oblast_name']   = districts['oblast_name'].str.strip()

districts['_key'] = districts['district_name'] + ' | ' + districts['oblast_name']

kz_geo = districts.set_index('_key')

print('Choose focal oblast from: ', sorted(districts['oblast_name'].unique()))

### Select focal oblast

In [ ]:
focal = 'Mangystau Region' 

if focal not in districts['oblast_name'].values:
    print(f"Oblast '{focal}' not found. Please choose from the list above.")

### Loading hazard data and concatenating historic with future datasets

In [ ]:
frames = []

for d in data:
    df_file = OUTPUT_DIR / f'droughthazard_KZ_{d}.csv'
    df = pd.read_csv(df_file)
    df['data'] = d
    frames.append(df)

df_ = pd.concat(frames, axis=0, ignore_index=True)

df_['oblast']        = df_['district'].str.split(' | ', n=1, regex=False).str[1].str.strip()
df_['district_name'] = df_['district'].str.split(' | ', n=1, regex=False).str[0].str.strip()

print(f'Loaded {len(df_)} rows across {len(data)} scenarios.')
print(f'Unique oblasts in hazard data: {sorted(df_["oblast"].dropna().unique())}')

### Subset hazard data to focal oblast

In [ ]:
focal_area    = df_['oblast'] == focal
df_focal_area = df_[focal_area][[
    'district', 'district_name', 'wasp_raw_mean', 'wasp_raw_q25',
    'wasp_raw_median', 'wasp_raw_q75', 'wasp_raw_count',
    'hazard_raw', 'data'
]].reset_index(drop=True)

n_dist = df_focal_area['district_name'].nunique()
print(f'Districts in focal oblast ({focal}): {n_dist}')

if n_dist == 0:
    print('WARNING: 0 districts found. Check that the focal oblast name exactly')
    print('matches one of the values printed in the shapefile cell above.')
    print('Available oblasts in hazard data:')
    print(sorted(df_['oblast'].dropna().unique()))

### Loading drought risk data and concatenating historic with future datasets

In [ ]:
risk_frames = []

for d in data:
    df_file = RISK_DIR / f'droughtrisk_KZ_{d}_districts.csv'
    df = pd.read_csv(df_file)
    df['data'] = d
    df['risk_cat'] = pd.to_numeric(df['risk_cat'], errors='coerce').astype(float)
    risk_frames.append(df)

df_r = pd.concat(risk_frames, axis=0, ignore_index=True)

df_r['oblast']        = df_r['district'].str.split(' | ', n=1, regex=False).str[1].str.strip()
df_r['district_name'] = df_r['district'].str.split(' | ', n=1, regex=False).str[0].str.strip()

df_r['data_label'] = (
    df_r['data']
    .str.replace('_nf', ', 2050', regex=False)
    .str.replace('_ff', ', 2080', regex=False)
)

print(f'Loaded {len(df_r)} rows across {len(data)} scenarios.')
print(f'Unique oblasts in risk data: {sorted(df_r["oblast"].dropna().unique())}')

## How does the absolute drought hazard (WASP value) for districts change in the future?

Compare the WASP values (median, q25 and q75) between districts for historic and future scenarios in the focal oblast.

In [ ]:
fig = go.Figure()

df_focal_area = df_focal_area.copy()
df_focal_area['wasp_raw_mean']   = df_focal_area['wasp_raw_mean'].abs()
df_focal_area['wasp_raw_median'] = df_focal_area['wasp_raw_median'].abs()
df_focal_area['wasp_raw_q75']    = df_focal_area['wasp_raw_q75'].abs()
df_focal_area['wasp_raw_q25']    = df_focal_area['wasp_raw_q25'].abs()

fig.add_trace(go.Bar(
    x=[df_focal_area['district_name'], df_focal_area['data']],
    y=df_focal_area['wasp_raw_median'],
    marker_color='#8CAED2',
    name='Median'
))

fig.add_trace(go.Scatter(
    x=[df_focal_area['district_name'], df_focal_area['data']],
    y=df_focal_area['wasp_raw_q25'],
    name='Quantile-25%',
    marker_color='#9cbd7e',
    mode='markers'
))

fig.add_trace(go.Scatter(
    x=[df_focal_area['district_name'], df_focal_area['data']],
    y=df_focal_area['wasp_raw_q75'],
    name='Quantile-75%',
    marker_color='#e4bace',
    mode='markers'
))

fig.update_layout(
    title=f'WASP Index values — {focal} — historic and future scenarios',
    xaxis=dict(tickangle=90, tickfont=dict(size=10))
)
fig.show()

## What is the relative drought risk in each district of Kazakhstan?

:::{note}
Risk categories are always relative to other districts in Kazakhstan — not absolute risk levels.
:::

In [ ]:
centroid = kz_geo.geometry.union_all().centroid
x_kz, y_kz = centroid.x, centroid.y

df_r['data_label'] = (
    df_r['data']
    .str.replace('_nf', ', 2050', regex=False)
    .str.replace('_ff', ', 2080', regex=False)
)

fig = px.choropleth_mapbox(
    df_r,
    geojson=kz_geo.geometry,
    locations='district',
    color='risk_cat',
    animation_frame='data_label',
    color_continuous_scale='reds',
    range_color=[1, 5],
    mapbox_style='open-street-map'
)

fig.update_layout(
    title='Current and projected drought risk — Kazakhstan districts',
    mapbox_center={'lat': y_kz, 'lon': x_kz},
    mapbox_zoom=4,
    height=700,
    coloraxis_colorbar=dict(
        title='Risk category',
        tickvals=[1, 2, 3, 4, 5],
        ticktext=['1', '2', '3', '4', '5']
    )
)
fig.show()

## How does the relative drought risk for districts change in the future?

Bar chart comparing raw risk scores for each district in the focal oblast across all scenarios.

:::{note}
Risk scores are relative to all Kazakhstan districts — not absolute risk levels.
:::

In [ ]:
print(f'Bar chart for historic and future relative drought risk — {focal}')

df_r_focal = df_r[df_r['oblast'] == focal].copy().reset_index(drop=True)

if df_r_focal.empty:
    print(f'WARNING: No risk data found for oblast "{focal}".')
    print('Check that focal matches a value in df_r["oblast"].')
else:
    fig4 = go.Figure()

    for dist_ in sorted(df_r_focal['district_name'].unique()):
        subset = df_r_focal[df_r_focal['district_name'] == dist_]
        fig4.add_trace(go.Bar(
            x=[subset['data_label'], subset['district_name']],
            y=subset['risk_raw'],
            name=dist_
        ))

    fig4.update_layout(
        title=f'Relative drought risk by scenario — {focal}',
        yaxis_title='Risk score (raw)',
        xaxis=dict(tickangle=45, tickfont=dict(size=10)),
        legend=dict(orientation='v', x=1.01)
    )
    fig4.show()

## Conclusions

This workflow visualises relative drought hazard and risk for Kazakhstan districts within a selected focal oblast. Changes in drought hazard and relative drought risk between districts can be compared across scenarios and timeframes.

Outputs from the hazard and risk assessment notebooks are used for the following scenarios:
- Historical (1981–2014)
- SSP1-2.6 near future (2031–2060)
- SSP1-2.6 far future (2071–2100)
- SSP3-7.0 near future (2031–2060)
- SSP3-7.0 far future (2071–2100)

## Contributors

This Kazakhstan adaptation was developed building on the original CLIMAAX workflow by [Silvia Artuso](https://iiasa.ac.at/staff/silvia-artuso) and [Dor Fridman](https://iiasa.ac.at/staff/dor-fridman) from [IIASA's Water Security Research Group](https://iiasa.ac.at/programs/biodiversity-and-natural-resources-bnr/water-security), supported by [Michaela Bachmann](https://iiasa.ac.at/staff/michaela-bachmann) from [IIASA's Systemic Risk and Resilience Research Group](https://iiasa.ac.at/programs/advancing-systems-analysis-asa/systemic-risk-and-resilience).